<a href="https://colab.research.google.com/github/PercyTechX/del-excel-al-ml/blob/main/temporada-1/05-mas-variables-y-la-temporada/solucion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Solución 1.5 · Más variables y la temporada

**Serie:** Del Excel al Machine Learning · **Temporada 1** · 🎬 Video próximamente

Estas son las respuestas del ejercicio. Si la tuya es distinta pero da el mismo resultado, ¡también está bien! En programación suele haber más de una forma de llegar a la respuesta.

### Prepara los datos

Ejecuta estas cuatro celdas. Ya dejan listas `entreno`, `prueba`, el `indice` de cada mes y la función `preparar_x`.

Algunos ejercicios usan variables de los anteriores, así que **ejecútalos en orden**.

In [1]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

DATOS = "https://raw.githubusercontent.com/PercyTechX/del-excel-al-ml/main/datos/"

In [2]:
def tabla_de_pronostico(ventas, clientes):
    """Una fila por cliente y mes, con cero si no compró, y lo que se sabía antes de ese mes."""
    meses = ventas["fecha"].str[:7].rename("mes")
    todos = pd.MultiIndex.from_product(
        [clientes["id_cliente"], sorted(meses.unique())], names=["id_cliente", "mes"]
    )
    compras = ventas.groupby(["id_cliente", meses])["total"].sum()
    tabla = compras.reindex(todos, fill_value=0).reset_index()

    por_cliente = tabla.groupby("id_cliente")["total"]
    tabla["mes_anterior"] = por_cliente.shift(1)
    tabla["promedio_3"] = por_cliente.transform(lambda s: s.shift(1).rolling(3).mean())
    tabla["promedio_6"] = por_cliente.transform(lambda s: s.shift(1).rolling(6).mean())
    tabla["numero_mes"] = tabla["mes"].str[5:].astype(int)

    columnas = ["id_cliente", "tipo", "ciudad", "id_vendedor", "fecha_alta"]
    tabla = pd.merge(tabla, clientes[columnas], on="id_cliente", how="left")
    tabla = tabla.dropna(subset=["promedio_6"])
    return tabla[tabla["mes"] + "-01" >= tabla["fecha_alta"]].drop(columns="fecha_alta")

In [3]:
def preparar_x(tabla, numeros, textos, columnas=None):
    """Convierte los textos en columnas de 0 y 1. Con `columnas`, usa las de entrenamiento."""
    x = pd.get_dummies(tabla[numeros + textos], columns=textos, dtype=float)
    if columnas is not None:
        x = x.reindex(columns=columnas, fill_value=0)
    return x

In [4]:
ventas = pd.read_csv(DATOS + "ventas.csv")
clientes = pd.read_csv(DATOS + "clientes.csv")

tabla = tabla_de_pronostico(ventas, clientes)

entreno = tabla[tabla["mes"] < "2025-07"]
prueba = tabla[tabla["mes"] >= "2025-07"].copy()

print(len(entreno), len(prueba))

ventas_2024 = ventas[ventas["fecha"].str[:4] == "2024"]
por_mes = ventas_2024.groupby(ventas_2024["fecha"].str[5:7].astype(int))["total"].sum()

indice = por_mes / por_mes.mean()

indice.round(2)

3475 1793


fecha
1     0.73
2     0.80
3     0.82
4     0.99
5     1.00
6     0.97
7     1.27
8     1.02
9     0.87
10    1.00
11    1.07
12    1.47
Name: total, dtype: float64

### 1. La regla con índice, por mes

Crea en `prueba` la columna `regla_indice`, como en la clase. El MAE debe dar `827`. Luego calcula su error por `numero_mes`, igual que en el 1.4. Diciembre debe dar cerca de `1028`. En el 1.4, sin índice, daba `1031`: cliente por cliente, casi no cambia. Veremos por qué en el ejercicio 4.

In [5]:
prueba["regla_indice"] = prueba["promedio_6"] * prueba["numero_mes"].map(indice)

print(round(mean_absolute_error(prueba["total"], prueba["regla_indice"])))

error = (prueba["total"] - prueba["regla_indice"]).abs()

print(error.groupby(prueba["numero_mes"]).mean().round())

827
numero_mes
7      871.0
8      794.0
9      706.0
10     768.0
11     793.0
12    1028.0
dtype: float64


### 2. El modelo

Entrena la regresión con `mes_anterior` y `promedio_3` como `numeros`, y `tipo`, `ciudad` y `numero_mes` como `textos`. Guarda su predicción en `prueba["regresion"]`. El MAE debe dar `921`.

In [6]:
numeros = ["mes_anterior", "promedio_3"]
textos = ["tipo", "ciudad", "numero_mes"]

x_entreno = preparar_x(entreno, numeros, textos)
x_prueba = preparar_x(prueba, numeros, textos, x_entreno.columns)

modelo = LinearRegression()
modelo.fit(x_entreno, entreno["total"])

prueba["regresion"] = modelo.predict(x_prueba)

print(round(mean_absolute_error(prueba["total"], prueba["regresion"])))

921


### 3. Una variable más

Crea otra lista, `numeros_6 = numeros + ["promedio_6"]`, y entrena otro modelo. No guardes su predicción en `prueba["regresion"]`, porque la usaremos después. Su MAE debe dar `893`. ¿Le gana a la regla con índice? Responde en un comentario.

In [7]:
numeros_6 = ["mes_anterior", "promedio_3", "promedio_6"]

x_entreno_6 = preparar_x(entreno, numeros_6, textos)
x_prueba_6 = preparar_x(prueba, numeros_6, textos, x_entreno_6.columns)

modelo_6 = LinearRegression()
modelo_6.fit(x_entreno_6, entreno["total"])

print(round(mean_absolute_error(prueba["total"], modelo_6.predict(x_prueba_6))))

# No: mejora a la otra regresión, pero la regla con índice sigue en 827.

893


### 4. Por cartera

Suma `total`, `regla_indice` y `regresion` por `id_vendedor` y `mes`. Calcula el MAE de cada método. Deben dar `4963` y `4819`.

¿Por qué el error de cada cliente casi no mejora, y el de la cartera sí? Responde en un comentario.

> 💡 Para ver dónde está la mejora, calcula el error de cartera por mes, con `error.groupby("mes").mean()`.

In [8]:
cartera = prueba.groupby(["id_vendedor", "mes"])[["total", "regla_indice", "regresion"]].sum()

print(round(mean_absolute_error(cartera["total"], cartera["regla_indice"])))
print(round(mean_absolute_error(cartera["total"], cartera["regresion"])))

error = (cartera["total"] - cartera["regla_indice"]).abs()
print(error.groupby("mes").mean().round())

# Cada cliente varía mucho por azar, y la temporada es una parte chica de ese error.
# Al sumar una cartera, el azar se compensa en parte y queda a la vista la temporada.

4963
4819
mes
2025-07    4917.0
2025-08    5699.0
2025-09    5117.0
2025-10    2648.0
2025-11    4080.0
2025-12    7316.0
dtype: float64


### 5. Los clientes sin compras

La regresión les predice compras a clientes que ya no compran. Crea una copia de la predicción que les ponga cero cuando `promedio_3` es 0:

```python
prueba["regresion_cero"] = prueba["regresion"].where(prueba["promedio_3"] > 0, 0)
```

`.where()` deja el valor si se cumple la condición, y si no, usa el 0. El MAE debe bajar a `861`. ¿Es trampa usar `promedio_3` aquí? Responde en un comentario.

In [9]:
prueba["regresion_cero"] = prueba["regresion"].where(prueba["promedio_3"] > 0, 0)

print(round(mean_absolute_error(prueba["total"], prueba["regresion_cero"])))

# No es trampa: promedio_3 mira los meses anteriores, y se conoce antes de predecir.

861


### 6. ¿Hacen falta todas?

Entrena el modelo sin `tipo` ni `ciudad`: solo `numero_mes` en `textos`. Debe dar `916` por cliente. ¿Te quedarías con este modelo? Responde en un comentario.

In [10]:
x_entreno_mes = preparar_x(entreno, numeros, ["numero_mes"])
x_prueba_mes = preparar_x(prueba, numeros, ["numero_mes"], x_entreno_mes.columns)

modelo_mes = LinearRegression()
modelo_mes.fit(x_entreno_mes, entreno["total"])

print(round(mean_absolute_error(prueba["total"], modelo_mes.predict(x_prueba_mes))))

# Sí: es más simple y rinde igual. Si una variable no aporta, se puede quitar.

916


## 🎉 ¡Bien hecho!

Nos vemos en el **episodio 1.6: Cuando el modelo se aprende las respuestas**.

---

*Del Excel al Machine Learning* · por [Percy Nazario](https://www.linkedin.com/in/percy-nazario/) · [Repositorio](https://github.com/PercyTechX/del-excel-al-ml) · Contenido bajo licencia CC BY 4.0